# 第11章 图像金字塔（跟随代码）

> 《OpenCV轻松入门：面向Python》第 11 章的跟随代码，逻辑与原书一致，仅把 lena 系列素材替换为本目录的 picture 素材。
> 运行前请确认工作目录为本文件夹（picture*.jpg 等素材都在这里）。


## 11.1 理论基础

- 图像金字塔：同一幅图像不同分辨率的子图集合，由原图不断向下采样产生。
- 向下采样两种方式：直接删除偶数行和偶数列；先滤波（邻域平均→平均金字塔，高斯滤波→高斯金字塔）再删。
- 向上采样：宽高各变为 2 倍，补零后用向下采样的高斯核滤波，核系数乘 4。
- 向上采样与向下采样不可逆。

## 11.2 pyrDown 函数及使用

`dst = cv2.pyrDown(src[, dstsize[, borderType]])`

- `src`：原始图像；`dst`：目标图像。
- `dstsize`：目标图像大小，默认 `Size((src.cols+1)/2, (src.rows+1)/2)`，须满足 |dst.width*2-src.cols|<=2、|dst.height*2-src.rows|<=2。
- `borderType`：边界类型，默认 BORDER_DEFAULT。

### 例 11.1 向下采样

原书素材为 lena.bmp（512×512 灰度），此处用 picture1_gray.jpg 顶替。

In [ ]:
import cv2
o=cv2.imread("picture1_gray.jpg",cv2.IMREAD_GRAYSCALE)
r1=cv2.pyrDown(o)
r2=cv2.pyrDown(r1)
r3=cv2.pyrDown(r2)
print("o.shape=",o.shape)
print("r1.shape=",r1.shape)
print("r2.shape=",r2.shape)
print("r3.shape=",r3.shape)
cv2.imshow("original",o)
cv2.imshow("r1",r1)
cv2.imshow("r2",r2)
cv2.imshow("r3",r3)
cv2.waitKey()
cv2.destroyAllWindows()


## 11.3 pyrUp 函数及使用

`dst = cv2.pyrUp(src[, dstsize[, borderType]])`

- `dstsize`：目标图像大小，默认 `Size(src.cols*2, src.rows*2)`，须满足 |dst.width-src.cols*2|<=dst.width mod 2、|dst.height-src.rows*2|<=dst.height mod 2。
- 内部过程：补零 → 用向下采样的高斯核滤波 → 核系数乘 4。

### 例 11.2 向上采样

原书素材为 lenas.bmp（64×64 彩色），此处用 picture3_mouth.jpg 顶替。

In [ ]:
import cv2
o=cv2.imread("picture3_mouth.jpg")
r1=cv2.pyrUp(o)
r2=cv2.pyrUp(r1)
r3=cv2.pyrUp(r2)
print("o.shape=",o.shape)
print("r1.shape=",r1.shape)
print("r2.shape=",r2.shape)
print("r3.shape=",r3.shape)
cv2.imshow("original",o)
cv2.imshow("r1",r1)
cv2.imshow("r2",r2)
cv2.imshow("r3",r3)
cv2.waitKey()
cv2.destroyAllWindows()


## 11.4 采样可逆性的研究

先向下采样、再向上采样（或反过来），尺寸恢复原状，但像素值并不一致——向上采样和向下采样不是互逆的。

### 例 11.3 先向下采样，再向上采样

原书彩色 lena 用 picture1.jpg 顶替。

In [ ]:
import cv2
o=cv2.imread("picture1.jpg")
down=cv2.pyrDown(o)
up=cv2.pyrUp(down)
diff=up-o   #构造diff图像，查看up与o的区别
print("o.shape=",o.shape)
print("up.shape=",up.shape)
cv2.imshow("original",o)
cv2.imshow("up",up)
cv2.imshow("difference",diff)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 11.4 先向上采样，再向下采样

In [ ]:
import cv2
o=cv2.imread("picture1.jpg")
up=cv2.pyrUp(o)
down=cv2.pyrDown(up)
diff=down-o   #构造diff图像，查看down与o的区别
print("o.shape=",o.shape)
print("down.shape=",down.shape)
cv2.imshow("original",o)
cv2.imshow("down",down)
cv2.imshow("difference",diff)
cv2.waitKey()
cv2.destroyAllWindows()


## 11.5 拉普拉斯金字塔

### 11.5.1 定义

拉普拉斯金字塔第 i 层：`Li = Gi - pyrUp(Gi+1)`，其中 Gi 是高斯金字塔第 i 层。

### 例 11.5 构造拉普拉斯金字塔

In [ ]:
import cv2
O=cv2.imread("picture1.jpg")
G0=O
G1=cv2.pyrDown(G0)
G2=cv2.pyrDown(G1)
G3=cv2.pyrDown(G2)
L0=G0-cv2.pyrUp(G1)
L1=G1-cv2.pyrUp(G2)
L2=G2-cv2.pyrUp(G3)
print("L0.shape=",L0.shape)
print("L1.shape=",L1.shape)
print("L2.shape=",L2.shape)
cv2.imshow("L0",L0)
cv2.imshow("L1",L1)
cv2.imshow("L2",L2)
cv2.waitKey()
cv2.destroyAllWindows()


### 11.5.2 应用

恢复关系式（由定义式移项得到）：`Gi = Li + pyrUp(Gi+1)`。

#### 例 11.6 恢复原始图像

In [ ]:
import cv2
import numpy as np
O=cv2.imread("picture1.jpg")
G0=O
G1=cv2.pyrDown(G0)
L0=O-cv2.pyrUp(G1)
RO=L0+cv2.pyrUp(G1)  # 通过拉普拉斯图像复原的原始图像
print("O.shape=",O.shape)
print("RO.shape=",RO.shape)
result=RO-O  # 将O和RO做减法运算
# 计算result的绝对值，避免求和时正负抵消，3+(-3)=0
result=abs(result)
# 计算result所有元素的和
print("原始图像O 与恢复图像RO 之差的绝对值和：",np.sum(result))


#### 例 11.7 恢复高斯金字塔内的多层图像

In [ ]:
import cv2
import numpy as np
O=cv2.imread("picture1.jpg")
#==============生成高斯金字塔===================
G0=O
G1=cv2.pyrDown(G0)
G2=cv2.pyrDown(G1)
G3=cv2.pyrDown(G2)
#============生成拉普拉斯金字塔===================
L0=G0-cv2.pyrUp(G1) #拉普拉斯金字塔第0层
L1=G1-cv2.pyrUp(G2) #拉普拉斯金字塔第1层
L2=G2-cv2.pyrUp(G3) #拉普拉斯金字塔第2层
#=================复原G0======================
RG0=L0+cv2.pyrUp(G1)  #通过拉普拉斯图像复原的原始图像G0
print("G0.shape=",G0.shape)
print("RG0.shape=",RG0.shape)
result=RG0-G0  #将RG0和G0相减
#计算result的绝对值，避免求和时正负抵消，3+(-3)=0
result=abs(result)
#计算result所有元素的和
print("原始图像G0 与恢复图像RG0 差值的绝对值和：",np.sum(result))
#=================复原G1======================
RG1=L1+cv2.pyrUp(G2) #通过拉普拉斯图像复原G1
print("G1.shape=",G1.shape)
print("RG1.shape=",RG1.shape)
result=RG1-G1  #将RG1和G1相减
print("原始图像G1 与恢复图像RG1 之差的绝对值和：",np.sum(abs(result)))
#=================复原G2======================
RG2=L2+cv2.pyrUp(G3) #通过拉普拉斯图像复原G2
print("G2.shape=",G2.shape)
print("RG2.shape=",RG2.shape)
result=RG2-G2  #将RG2和G2相减
print("原始图像G2 与恢复图像RG2 之差的绝对值和：",np.sum(abs(result)))
